# Real-Time Retail Feedback Intelligence
### Generative AI for Customer Review Analysis using Prompt Engineering

[Python](https://python.org)
[OpenAI](https://openai.com)
[Prompt Engineering]

---

**Author:** Luisa Correa  
**Domain:** NLP · Generative AI · Retail Analytics  
**Dataset:** [Women's E-Commerce Clothing Reviews](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews)


## Table of Contents

1. [Business Context & Objective](#1-business-context--objective)
2. [Setup & Dependencies](#2-setup--dependencies)
3. [Data Loading & Overview](#3-data-loading--overview)
4. [Data Cleaning & Preprocessing](#4-data-cleaning--preprocessing)
5. [Exploratory Data Analysis (EDA)](#5-exploratory-data-analysis)
6. [Building the GenAI Pipeline](#6-building-the-genai-pipeline)
   - 6.1 [LLM Client Setup](#61-llm-client-setup)
   - 6.2 [Helper Functions](#62-helper-functions)
   - 6.3 [Zero-Shot Prompting](#63-zero-shot-prompting)
   - 6.4 [Few-Shot Prompting](#64-few-shot-prompting)
   - 6.5 [Chain-of-Thought (CoT) Prompting](#65-chain-of-thought-cot-prompting)
7. [Recommendation Prediction](#7-recommendation-prediction)
8. [Results & Technique Comparison](#8-results--technique-comparison)
9. [Actionable Recommendations](#9-actionable-recommendations)
10. [Conclusion](#10-conclusion)


## 1. Business Context & Objective

**ChicStyle** is a growing fashion retail platform that experiences massive spikes in customer activity during festive seasons and holiday sales. As the volume of incoming reviews increases dramatically, even a slight delay in reading or responding to customer feedback can have serious consequences — leading to frustrated customers, reduced brand trust, and lost repeat business.

### Problem
Traditional NLP models struggle with **complex or mixed feedback**. For example, in:

> *"The fit is great but the color was not as per the product image"*

older systems assign a single sentiment, missing the nuanced positive/negative split. Retailers need more granular, actionable insights.

### Solution
This notebook builds a **Generative AI feedback intelligence system** using prompt engineering techniques to:

| Capability | Description |
|---|---|
| Sentiment Analysis | Classify each review as Positive, Neutral, or Negative |
| Category Detection | Identify which product/service the feedback refers to |
| Urgency Scoring | Flag high-priority issues requiring immediate action |
| Personalized Messaging | Generate auto-replies tailored to each customer's sentiment |
| Retail Insights | Surface actionable recommendations for product & ops teams |

### Dataset
The **Women's E-Commerce Clothing Reviews** dataset contains 23,486 real customer reviews with the following columns:

| Column | Description |
|---|---|
| `Clothing.ID` | Unique product identifier |
| `Age` | Reviewer age |
| `Title` | Review title |
| `Review.Text` | Full review text |
| `Rating` | Score 1–5 (1=Worst, 5=Best) |
| `Recommended.IND` | 1 if customer recommends, 0 otherwise |
| `Positive.Feedback.Count` | Number of helpful votes from other customers |
| `Division.Name` | High-level product division |
| `Department.Name` | Product department |


## 2. Setup & Dependencies

In [ ]:
# Install required libraries (run once)
%pip install -q pandas matplotlib seaborn openai wordcloud plotly tqdm scikit-learn

In [ ]:
import re
import json
import time
from typing import Dict, List, Optional, Any, Tuple

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from wordcloud import WordCloud
from tqdm import tqdm
from IPython.display import display

import openai
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")
print("All libraries imported successfully")

## 3. Data Loading & Overview

In [ ]:
# ── Load the dataset ──
# Update the path below to point to your local copy of the dataset
DATA_PATH = "data/Dataset_-_Real-Time_Retail_Feedback_Intelligence.csv"

df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Dataset shape: {df.shape}")
display(df.head())

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Descriptive statistics for numeric columns
df[['Age', 'Rating', 'Positive.Feedback.Count']].describe().round(2)

**Key observations:**
- The dataset has **23,486 reviews** with 10 columns.
- `Title` and `Review.Text` have missing values — we will handle these in preprocessing.
- Numeric columns have expected ranges (Age, Rating, Positive.Feedback.Count).


## 4. Data Cleaning & Preprocessing

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

`Review.Text` has ~3.6% missing values. Rather than dropping these rows immediately,
we first combine `Title` + `Review.Text` into a single `title_and_text` field — this way
reviews with only a title (but no body text) can still contribute signal to the model.
Rows where *both* are empty are then dropped.


In [ ]:
# Fill NaN with empty string before concatenating
df['Title'] = df['Title'].fillna('')
df['Review.Text'] = df['Review.Text'].fillna('')

# Combine title and review text for richer context
df['title_and_text'] = (
    df['Title'].str.strip() + ' ' + df['Review.Text'].str.strip()
).str.strip()

# Drop rows where the combined text is empty
to_drop = df[df['title_and_text'].str.strip() == ''].index
print(f"Rows dropped (no text at all): {len(to_drop)}")
df = df.drop(to_drop)
print(f"Final dataset shape: {df.shape}")

## 5. Exploratory Data Analysis

### 5.1 Rating Distribution

In [ ]:
fig = px.histogram(
    df, x='Rating',
    title='Distribution of Product Ratings',
    color_discrete_sequence=['steelblue'],
    labels={'Rating': 'Rating (1=Worst, 5=Best)', 'count': 'Number of Reviews'}
)
fig.update_layout(bargap=0.1)
fig.show()

Ratings are strongly **right-skewed** — most reviews are 4 or 5 stars, reflecting an overall positive customer base.
This also means the model needs to be sensitive to **negative signals buried in positive-leaning reviews**,
which is a key motivation for using Chain-of-Thought prompting.


### 5.2 Word Clouds: Positive vs. Negative Reviews

In [ ]:
positive_rev = " ".join(df[df['Rating'] >= 4]['title_and_text'])
negative_rev = " ".join(df[df['Rating'] <= 2]['title_and_text'])

wordcloud_pos = WordCloud(width=800, height=400, background_color='white').generate(positive_rev)
wordcloud_neg = WordCloud(width=800, height=400, background_color='white', colormap='Reds').generate(negative_rev)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.imshow(wordcloud_pos, interpolation='bilinear')
ax1.set_title('Common Words in Positive Reviews (4–5)', fontsize=14, pad=12)
ax1.axis('off')
ax2.imshow(wordcloud_neg, interpolation='bilinear')
ax2.set_title('Common Words in Negative Reviews (1–2)', fontsize=14, pad=12)
ax2.axis('off')
plt.tight_layout()
plt.show()

Positive reviews center on **"love," "fit," "comfortable," "perfect"** — customers are happy when products fit well and feel good.

Negative reviews feature **"size," "small," "ordered," "back"** — pointing to sizing issues, unmet expectations, and returns.

Importantly, words like **"fit"** and **"dress"** appear in *both* clouds — the same attributes that delight customers also frustrate them when they go wrong.
This reinforces why star ratings alone are insufficient: a detailed text analysis is needed to capture the full picture.


### 5.3 Recommendation Rate by Department

In [ ]:
rec_rate = (
    df.groupby("Department.Name")["Recommended.IND"]
    .mean()
    .sort_values(ascending=True)
    .reset_index()
)
rec_rate.columns = ["Department", "Recommendation Rate"]

fig = px.bar(
    rec_rate, x="Department", y="Recommendation Rate",
    title="Recommendation Rate by Department",
    text_auto=".2f",
    color="Recommendation Rate",
    color_continuous_scale="Blues"
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()

The **Trend** department has the lowest recommendation rate (~74%), significantly below other departments.
This is an actionable insight — the Trend line may need product quality review or better expectation-setting in product descriptions.


### 5.4 Rating vs. Recommendation Alignment

In [ ]:
rec_by_rating = (
    df.groupby('Rating')['Recommended.IND']
    .value_counts(normalize=True)
    .mul(100).round(1)
    .rename('Percentage')
    .reset_index()
)
rec_by_rating['Recommended.IND'] = rec_by_rating['Recommended.IND'].map({1: 'Recommended', 0: 'Not Recommended'})

fig = px.bar(
    rec_by_rating, x='Rating', y='Percentage', color='Recommended.IND',
    title='% Recommended vs. Not Recommended by Rating',
    barmode='stack',
    color_discrete_map={'Recommended': 'steelblue', 'Not Recommended': 'tomato'}
)
fig.show()

There is a strong alignment between Rating and Recommendation — *except* at **3 stars**, where the majority of customers do **not** recommend the product.
This suggests 3-star reviews should be treated more like negative feedback from a business action standpoint,
and is something the GenAI model should account for in its urgency classification.


### 5.5 Key EDA Takeaways

| Finding | Business Implication |
|---|---|
| Ratings skew 4–5 | Most feedback is positive; negative signals are subtle |
| 3-star ≈ negative behavior | Treat neutral ratings as potential action items |
| Fit & sizing dominate complaints | Priority for product design and size guides |
| "Trend" dept. underperforms | Needs quality or presentation review |
| High-engagement negative reviews exist | Some issues resonate broadly — prioritize these |

These insights directly inform the **prompt design**, particularly for urgency sensitivity and insight generation.


## 6. Building the GenAI Pipeline

### 6.1 LLM Client Setup

> **API Key Setup:** Store your OpenAI API key as an environment variable for security.
> In Colab: use the *Secrets* panel (left sidebar). In a local environment: use a `.env` file with `python-dotenv`.


In [ ]:
import os

# ── Option A: Colab Secrets (recommended in Colab) ──
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    print("API key loaded from Colab Secrets")

# ── Option B: Environment variable (local / GitHub Codespaces) ──
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    if OPENAI_API_KEY:
        print("API key loaded from environment variable")
    else:
        raise ValueError("No API key found. Set OPENAI_API_KEY as a Colab Secret or environment variable.")

client = openai.OpenAI(api_key=OPENAI_API_KEY)
MODEL_NAME = "gpt-4o-mini"
print(f"Model: {MODEL_NAME}")

### Sample Selection

We analyze a **sample of 50 reviews** to balance cost and statistical coverage across all prompting techniques.

> **Cost tip:** Start with 5–10 reviews during prompt development, then scale to 50 for final evaluation.


In [ ]:
# Reproducible sample of 50 reviews
df_sample = df.sample(n=50, random_state=1).copy().reset_index(drop=True)
print(f"Sample shape: {df_sample.shape}")
df_sample[['title_and_text', 'Rating', 'Recommended.IND', 'Department.Name']].head()

---
### 6.2 Helper Functions

The pipeline uses three shared utility functions:

1. **`structured_output`** — calls the LLM and parses the response into a structured dict
2. **`generate_output`** — thin wrapper around `structured_output`
3. **`judge_output`** — LLM-as-a-judge to score each prediction on a 0–1 scale

Using an **LLM-as-Judge** is appropriate here because tasks like summarization and insight generation are *subjective* — there is no ground-truth label. The judge evaluates dimensions including sentiment accuracy, summary faithfulness, message empathy, and insight actionability.


In [ ]:
def structured_output(review_text: str, prompt: str) -> dict:
    """
    Calls the LLM with the given prompt and parses the structured response.
    Returns a dict with keys: Category, Sentiment, Urgency_Level, Summary,
    Personalized_Message, Insight, Raw_Output.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,  # deterministic for consistent evaluation
            messages=[
                {
                    "role": "system",
                    "content": "You are a precise assistant that extracts structured information from customer reviews."
                },
                {
                    "role": "user",
                    "content": prompt.format(review_text=review_text)
                }
            ]
        )
    except Exception as e:
        error_msg = f"API_ERROR: {type(e).__name__}: {e}"
        return {k: error_msg if k in ("Summary", "Raw_Output") else "" for k in
                ["Category", "Sentiment", "Urgency_Level", "Summary",
                 "Personalized_Message", "Insight", "Raw_Output"]}

    text = response.choices[0].message.content or ""
    lines = [ln.strip() for ln in text.strip().splitlines() if ln.strip()]

    out = {
        "Category": "", "Sentiment": "", "Urgency_Level": "",
        "Summary": "", "Personalized_Message": "", "Insight": "",
        "Raw_Output": text
    }

    for line in lines:
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        key = key.strip().lower().replace(" ", "_")
        value = value.strip()
        if key.startswith("category"):          out["Category"] = value
        elif key.startswith("sentiment"):       out["Sentiment"] = value
        elif key.startswith("urgency_level"):   out["Urgency_Level"] = value
        elif key.startswith("summary"):         out["Summary"] = value
        elif key.startswith("personalized"):    out["Personalized_Message"] = value
        elif key.startswith("insight"):         out["Insight"] = value

    return out


def generate_output(review_text: str, prompt: str) -> dict:
    """Wrapper around structured_output for consistency."""
    return structured_output(review_text, prompt)

In [ ]:
# ── LLM-as-a-Judge ──
JUDGE_PROMPT = r"""
You are an expert evaluator of AI-generated outputs for a retail feedback analysis system.

The goal of the system is to help retailers take action on customer issues, improve product quality,
and enhance customer satisfaction.

Evaluate the model output based on the original review across these criteria:

1. Category correctness – Does the category reflect the main issue accurately?
2. Sentiment accuracy – Is it correctly classified as Positive, Neutral, or Negative?
   (Preferably overestimate issues rather than miss a negative experience.)
3. Urgency level – Does it correctly reflect the need for action?
4. Summary quality – Is it faithful, clear, and concise?
5. Personalized message – Is it appropriate, empathetic, and directly usable?
6. Insight actionability – Is it specific and actionable for the retail team?

Scoring:
- 0.0 = poor (not usable)
- 0.5 = partially correct (some useful elements but issues remain)
- 1.0 = excellent (accurate, consistent, and fully actionable)

Return ONLY a single number between 0 and 1 (e.g. 0.75).

Review:
{review}

Model Output:
---
{output}
---
"""


def judge_output(review_text: str, output_text: str) -> Optional[float]:
    """
    Uses an LLM-as-judge to evaluate output quality.
    Returns a float score in [0, 1], or None on failure.
    """
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,
            messages=[
                {"role": "system", "content": "You are a strict and consistent evaluator. Return only a numeric score between 0 and 1."},
                {"role": "user",   "content": JUDGE_PROMPT.format(review=review_text, output=output_text)}
            ]
        )
        score_str = response.choices[0].message.content.strip()
        match = re.search(r"\d*\.?\d+", score_str)
        return float(match.group()) if match else None
    except Exception:
        return None


def run_and_score(prompt: str, label: str, df_src: pd.DataFrame) -> pd.DataFrame:
    """
    Runs a prompt over all reviews, evaluates each output with the judge,
    and returns a DataFrame with prefixed columns.
    """
    outputs = [generate_output(r, prompt) for r in tqdm(df_src["title_and_text"], desc=f"{label} → Generating")]
    out_df = pd.DataFrame(outputs).add_prefix(f"{label}_")

    scores = [
        judge_output(review, output)
        for review, output in tqdm(
            zip(df_src["title_and_text"], out_df[f"{label}_Raw_Output"]),
            total=len(df_src), desc=f"{label} → Judging"
        )
    ]
    out_df[f"{label}_Score"] = scores
    avg = pd.Series(scores).dropna().mean()
    print(f"\n{label} — Average Score: {avg:.3f}\n")
    return out_df

print("Helper functions defined")

---
### 6.3 Zero-Shot Prompting

**Zero-Shot prompting** gives the model only a task description — no examples.
The model must infer how to respond purely from the instruction.

We test two versions:
- **V1**: Minimal instruction — just the output format
- **V2**: Enriched with business context, urgency definitions, and message guidelines


In [ ]:
ZERO_SHOT_V1 = """
Analyze the following customer review and provide:

Category: <value>
Sentiment: <Positive, Neutral, or Negative>
Urgency_Level: <Low, Medium, or High>
Summary: <brief summary>
Personalized_Message: <short message for customer>
Insight: <useful comment for the retail team>

Review: {review_text}
"""

zs_v1_df = run_and_score(ZERO_SHOT_V1, "ZS_V1", df_sample)
df_sample = pd.concat([df_sample, zs_v1_df], axis=1)

In [ ]:
ZERO_SHOT_V2 = """
You are an AI assistant for the retail company ChicStyle. Analyze the following customer review
with high accuracy, as the output will be used for customer support and business decision-making.

Provide the response in this exact format:

Category: <value>
Sentiment: <Positive, Neutral, or Negative>
Urgency_Level: <Low, Medium, or High>
Summary: <brief and accurate summary>
Personalized_Message: <short customer message>
Insight: <useful and actionable comment for the retail team>

Guidelines:
- Base your answer only on the review text; do not add information not present in the review.
- Urgency_Level reflects how quickly the issue should be addressed:
  - Low: positive feedback or minor comments
  - Medium: moderate issues or mixed feedback
  - High: strong dissatisfaction, damaged items, or issues requiring immediate follow-up
- Personalized_Message must be sentiment-based and directly usable:
  - Positive → thank the customer
  - Neutral  → acknowledge the feedback
  - Negative → apologize and indicate follow-up
- Insight must be specific and actionable for the retail team.

Review: {review_text}
"""

zs_v2_df = run_and_score(ZERO_SHOT_V2, "ZS_V2", df_sample)
df_sample = pd.concat([df_sample, zs_v2_df], axis=1)

**Zero-Shot Observations:**

V2 scored slightly lower than V1 (0.810 vs. 0.833) by the judge — but this is misleading.
V2 produces **more specific and useful results**: it correctly labels a review as "Sizing" rather than just "Clothing",
and writes customer messages that reflect actual sentiment rather than generic responses.

The score gap on 50 reviews is well within sampling variance. What V2 adds (business context, urgency definitions, message guidelines)
makes it significantly more useful in a real retail setting.


---
### 6.4 Few-Shot Prompting

**Few-Shot prompting** provides the model with labeled examples before the actual task.
This helps the model understand the expected format and reasoning style through demonstration.


In [ ]:
FEW_SHOT_V1 = """
Analyze the following customer review and provide:

Category: <value>
Sentiment: <Positive, Neutral, or Negative>
Urgency_Level: <Low, Medium, or High>
Summary: <brief summary>
Personalized_Message: <short message for customer>
Insight: <useful comment for the retail team>

Use the same format as the examples below.

Example 1
Review: The fabric feels great and the dress fits perfectly. Very comfortable to wear.
Category: Product Quality
Sentiment: Positive
Urgency_Level: Low
Summary: The customer is very satisfied with the product quality, comfort, and fit.
Personalized_Message: Thank you for your positive feedback! We are glad to hear you are happy with the fit and comfort.
Insight: High-quality materials and good fit contribute strongly to customer satisfaction.

Example 2
Review: My package arrived late and the box was damaged, although the item inside was fine.
Category: Delivery
Sentiment: Negative
Urgency_Level: High
Summary: The customer is unhappy due to late delivery and damaged packaging despite the product being intact.
Personalized_Message: We are sorry for the inconvenience. A team member will reach out soon to assist you.
Insight: Improving delivery reliability and packaging quality can reduce negative customer experiences.

Example 3
Review: I had to contact customer service for a return, and although they were polite, the process took too long.
Category: Returns
Sentiment: Neutral
Urgency_Level: Medium
Summary: The customer experienced delays in the return process despite polite customer service.
Personalized_Message: We appreciate your feedback and will work to improve the efficiency of our return process.
Insight: Faster response times and streamlined return processes can improve overall customer satisfaction.

Now analyze this review:
Review: {review_text}
"""

fs_v1_df = run_and_score(FEW_SHOT_V1, "FS_V1", df_sample)
df_sample = pd.concat([df_sample, fs_v1_df], axis=1)

In [ ]:
FEW_SHOT_V2 = """
You are an AI assistant for retail company ChicStyle. Your task is to analyze customer reviews
accurately, as the output will be used for customer support and business decision-making.

For each review, provide a response in this exact format:

Category: <value>
Sentiment: <Positive, Neutral, or Negative>
Urgency_Level: <Low, Medium, or High>
Summary: <brief and accurate summary>
Personalized_Message: <short customer message>
Insight: <useful and actionable comment for the retail team>

Rules:
- Base your answer only on the review text; do not add information not present.
- Urgency_Level:
  - Low: positive or minor feedback
  - Medium: moderate or mixed feedback
  - High: urgent issues (defects, wrong items, strong dissatisfaction)
- Sentiment must be exactly one of: Positive, Neutral, or Negative.
- Personalized_Message must be directly usable:
  - Positive → thank the customer
  - Neutral  → acknowledge the feedback
  - Negative → apologize and indicate a follow-up
- Insight must name a specific, actionable improvement.

Examples:

Review: The fabric feels great and the dress fits perfectly. Very comfortable to wear.
Category: Product Quality
Sentiment: Positive
Urgency_Level: Low
Summary: Customer is satisfied with fabric quality, comfort, and fit.
Personalized_Message: Thank you for your wonderful feedback! We're delighted to hear the dress fits and feels great.
Insight: Positive fabric and fit experiences are key drivers of satisfaction — maintain quality in these areas.

Review: I received the wrong size. Very disappointed, had to return it immediately.
Category: Fulfillment
Sentiment: Negative
Urgency_Level: High
Summary: Customer received incorrect size and returned the product due to fulfillment error.
Personalized_Message: We sincerely apologize for the inconvenience. A team member will contact you shortly to resolve this.
Insight: Fulfillment accuracy needs review — incorrect size shipments are a critical driver of returns and dissatisfaction.

Review: The blouse is pretty, but the sizing runs small. I exchanged for a larger size.
Category: Sizing
Sentiment: Neutral
Urgency_Level: Medium
Summary: Customer finds the blouse attractive but notes inconsistent sizing requiring an exchange.
Personalized_Message: Thank you for your feedback. We're sorry for the sizing inconvenience and are working to improve consistency.
Insight: Consistent sizing issues across products suggest the need for updated size charts and customer guidance.

Now analyze this review:
Review: {review_text}
"""

fs_v2_df = run_and_score(FEW_SHOT_V2, "FS_V2", df_sample)
df_sample = pd.concat([df_sample, fs_v2_df], axis=1)

**Few-Shot Observations:**

Both V1 and V2 scored 0.845 — the examples in V1 already anchored the model well.
V2 adds more structured rules without significant score improvement in this sample,
but provides better consistency in edge cases and is more suitable for production.


---
### 6.5 Chain-of-Thought (CoT) Prompting

**Chain-of-Thought prompting** instructs the model to reason step-by-step *internally* before producing the output.
The reasoning is not shown — only the final structured answer is returned.

This approach is particularly effective for **ambiguous or mixed-sentiment reviews**, where the surface-level tone
doesn't reflect the underlying customer issue.


In [ ]:
COT_V1 = """
Analyze the following customer review.

Think step by step internally about:
- the main issue or topic
- the sentiment
- the urgency
- the key summary points
- the customer message
- the business insight

Do not show your reasoning.

Return only the final answer in this format:

Category: <value>
Sentiment: <Positive, Neutral, or Negative>
Urgency_Level: <Low, Medium, or High>
Summary: <brief summary>
Personalized_Message: <short message for customer>
Insight: <useful comment for the retail team>

Review: {review_text}
"""

cot_v1_df = run_and_score(COT_V1, "CoT_V1", df_sample)
df_sample = pd.concat([df_sample, cot_v1_df], axis=1)

In [ ]:
COT_V2 = """
You are an AI assistant for the retail company ChicStyle. Your output will be used for customer
support and business decision-making, so accuracy, consistency, and actionability are critical.

Analyze the following customer review.

Think step by step internally using this process:
1. Identify the main product, service, or issue mentioned
2. Determine whether the overall sentiment is Positive, Neutral, or Negative
3. Assess whether the issue requires Low, Medium, or High urgency
4. Extract the main point for a concise and faithful summary
5. Create a customer response appropriate to the sentiment
6. Identify one useful and actionable insight for the retail team

Perform this reasoning internally only. Do not include reasoning in the final answer.

Return the final answer in this exact format:

Category: <value>
Sentiment: <Positive, Neutral, or Negative>
Urgency_Level: <Low, Medium, or High>
Summary: <brief and accurate summary>
Personalized_Message: <short customer message>
Insight: <useful and actionable comment for the retail team>

Rules:
- Base your answer only on the review text
- Sentiment must be exactly one of: Positive, Neutral, or Negative
- Keep the summary concise and faithful to the review
- Urgency_Level:
  - Low: no action needed or minor feedback
  - Medium: moderate issue that should be reviewed
  - High: issue requiring immediate attention or follow-up
- Personalized_Message:
  - Positive → thank the customer
  - Neutral  → acknowledge the feedback
  - Negative → apologize and indicate follow-up
- Insight must be specific, actionable, and suggest what to improve or investigate

Review: {review_text}
"""

cot_v2_df = run_and_score(COT_V2, "CoT_V2", df_sample)
df_sample = pd.concat([df_sample, cot_v2_df], axis=1)

**CoT Observations:**

CoT V2 achieved the **highest score of 0.903** — the top performer across all techniques.

Unlike Zero-Shot and Few-Shot, CoT consistently identified nuanced dissatisfaction, especially in reviews that combine
positive sentiment with specific complaints. It proved more sensitive to borderline cases, making it the most suitable
approach for production where missing a dissatisfied customer carries higher business risk than reviewing an extra case.


## 7. Recommendation Prediction

Beyond sentiment, we test whether the GenAI model can predict `Recommended.IND` from review text alone.
This is evaluated against the ground-truth label using accuracy and a classification report.


In [ ]:
RECOMMEND_PROMPT = """
Based only on the customer review below, predict whether the customer would recommend the product.

Review: {review_text}

Respond in this exact format:
Predicted_Recommended: <Yes or No>
Reasoning: <one sentence explanation>
"""

def get_recommendation_output(review_text: str) -> str:
    """Calls the LLM for a recommendation prediction."""
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME, temperature=0,
            messages=[
                {"role": "system", "content": "You are an expert at inferring customer satisfaction from review text."},
                {"role": "user",   "content": RECOMMEND_PROMPT.format(review_text=review_text)}
            ]
        )
        return response.choices[0].message.content or ""
    except Exception as e:
        return f"ERROR: {e}"


def parse_recommendation_output(text: str) -> dict:
    """Parses LLM output into structured recommendation fields."""
    out = {"Predicted_Recommended_IND": None, "Reason": ""}
    for line in text.strip().splitlines():
        if ":" not in line:
            continue
        key, value = line.split(":", 1)
        key, value = key.strip().lower(), value.strip()
        if key.startswith("predicted"):
            out["Predicted_Recommended_IND"] = 1 if "yes" in value.lower() else 0
        elif key.startswith("reasoning"):
            out["Reason"] = value
    return out


df_rec = df_sample.copy().reset_index(drop=True)
results = [
    parse_recommendation_output(get_recommendation_output(review))
    for review in tqdm(df_rec["title_and_text"], desc="Recommendation Predictions")
]

df_rec["Predicted_Recommended_IND"] = [r["Predicted_Recommended_IND"] for r in results]
df_rec["Recommendation_Reason"]     = [r["Reason"] for r in results]

In [ ]:
# Evaluate against ground truth
valid = df_rec.dropna(subset=["Predicted_Recommended_IND"]).copy()
y_true = valid["Recommended.IND"].astype(int)
y_pred = valid["Predicted_Recommended_IND"].astype(int)

accuracy = accuracy_score(y_true, y_pred)
print(f"\nRecommendation Prediction Accuracy: {accuracy:.2%}\n")
print(classification_report(y_true, y_pred, target_names=["Not Recommended", "Recommended"]))

The model achieves **~86% accuracy** on recommendation prediction — strong performance given it relies solely on unstructured text.

Mismatches typically occur in reviews where customers express satisfaction *despite* experiencing friction (e.g., size issues that were resolved). This highlights a key insight: **recommendation behavior alone is not a reliable signal for identifying operational problems** — deeper review analysis is necessary.


## 8. Results & Technique Comparison

### 8.1 Sentiment Distribution by Technique

In [ ]:
def plot_sentiment_distribution(df, col, title):
    order = ["Positive", "Neutral", "Negative"]
    counts = df[col].value_counts().reindex(order, fill_value=0)
    colors = {"Positive": "steelblue", "Neutral": "gold", "Negative": "tomato"}

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
    ax.set_title(title, fontsize=12, pad=10)
    ax.set_ylabel("Count")
    ax.set_ylim(0, counts.max() + 5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                int(bar.get_height()), ha='center', va='bottom', fontsize=10)
    plt.tight_layout()
    plt.show()

for col, label in [
    ("ZS_V2_Sentiment",  "Zero-Shot V2"),
    ("FS_V2_Sentiment",  "Few-Shot V2"),
    ("CoT_V2_Sentiment", "Chain-of-Thought V2"),
]:
    plot_sentiment_distribution(df_sample, col, f"Sentiment Distribution — {label}")

### 8.2 Judge Score Summary

In [ ]:
score_summary = pd.DataFrame({
    "Technique": ["Zero-Shot V1", "Zero-Shot V2", "Few-Shot V1", "Few-Shot V2", "CoT V1", "CoT V2"],
    "Avg Score": [
        df_sample["ZS_V1_Score"].mean(),
        df_sample["ZS_V2_Score"].mean(),
        df_sample["FS_V1_Score"].mean(),
        df_sample["FS_V2_Score"].mean(),
        df_sample["CoT_V1_Score"].mean(),
        df_sample["CoT_V2_Score"].mean(),
    ]
}).round(3)

fig = px.bar(
    score_summary, x="Technique", y="Avg Score",
    title="LLM-as-Judge: Average Score by Prompting Technique",
    color="Avg Score", color_continuous_scale="Greens",
    text_auto=".3f"
)
fig.update_layout(yaxis_range=[0.7, 1.0])
fig.show()

score_summary.sort_values("Avg Score", ascending=False)

### 8.3 Cross-Technique Sentiment Agreement

We compare the V2 versions of each technique to identify where they agree and where they diverge — especially on borderline cases.


In [ ]:
# How often do all three techniques agree?
agree_mask = (
    (df_sample["ZS_V2_Sentiment"] == df_sample["FS_V2_Sentiment"]) &
    (df_sample["FS_V2_Sentiment"] == df_sample["CoT_V2_Sentiment"])
)
print(f"Full agreement across techniques: {agree_mask.sum()}/50 reviews ({agree_mask.mean():.0%})")

# Show disagreements
disagreements = df_sample[~agree_mask][["title_and_text", "ZS_V2_Sentiment", "FS_V2_Sentiment", "CoT_V2_Sentiment"]].copy()
disagreements.columns = ["Review", "Zero-Shot", "Few-Shot", "CoT"]
print(f"\nDisagreements ({len(disagreements)} reviews):")
display(disagreements)

**Key finding:** Disagreements are concentrated in **borderline neutral/negative** cases.
CoT V2 tends to flag more of these as Negative, while Zero-Shot and Few-Shot classify them as Neutral.
Given that missing a dissatisfied customer is costlier than reviewing one extra case, **CoT's sensitivity is a feature, not a bug**.


## 9. Actionable Recommendations

Based on the GenAI pipeline analysis, here are concrete recommendations for ChicStyle:

### Short-Term (3–6 months)

| Action | Rationale |
|---|---|
| Update size guides with customer fit feedback (e.g., "runs small/large") | Top driver of neutral/negative reviews |
| Improve fulfillment accuracy checks | Incorrect sizes shipped is a High-urgency pattern |
| Create a real-time escalation workflow for High-urgency reviews | CoT successfully flags these; now act on them |

### Long-Term (6–12 months)

| Action | Rationale |
|---|---|
| Standardize sizing across product lines | Reduces cross-product inconsistency complaints |
| Use aggregated fit/comfort feedback to guide product design | Recurring discomfort patterns in reviews suggest design opportunities |
| Integrate the GenAI pipeline into product, ops, and CX decision workflows | Automates the full feedback-to-action loop |

### Why GenAI Outperforms Traditional NLP

This pipeline creates value by:
- **Reducing manual effort** — automatically classifies, summarizes, and generates structured output at scale
- **Enabling real-time issue detection** — identifies dissatisfaction early, even in mixed reviews
- **Improving decision quality** — provides specific, actionable insights that teams can act on immediately
- **Enhancing customer experience** — supports timely, personalized responses that build loyalty


## 10. Conclusion

This project demonstrates that **Generative AI can effectively convert unstructured customer feedback into structured, actionable intelligence at scale**.

### Model Performance Summary

| Technique | Avg Score | Best For |
|---|---|---|
| Zero-Shot V1 | 0.833 | Baseline |
| Zero-Shot V2 | 0.810 | Business-aware baseline |
| Few-Shot V1 | 0.845 | Format consistency |
| Few-Shot V2 | 0.845 | Production-ready format |
| **CoT V1** | **—** | Intermediate |
| **CoT V2** | **0.903** | **Recommended for production** |

### Key Findings

1. **CoT prompting is most effective** for nuanced, mixed-sentiment reviews — it surfaces issues that other techniques smooth over.
2. **Customer recommendation behavior ≠ customer satisfaction** — many customers recommend products despite friction, making deeper review analysis essential.
3. **Sizing and fit are the dominant pain points** across all departments, with a clear opportunity for product improvement.
4. **Rating alone is insufficient** — 3-star reviews behave like negative feedback and require active follow-up.

### Production Recommendation

Deploy **Chain-of-Thought V2** as the primary analysis engine. While it flags slightly more cases as negative, this sensitivity provides earlier intervention opportunities. The cost of missing a dissatisfied customer far outweighs the cost of reviewing one additional case.

---

*This project was built as part of the [MIT & Great Learning](https://www.greatlearning.in/) Applied AI & DS program capstone.*
